# えっちな動画を作る（MiniMax H3）

写真なしの文章から、または写真1枚から、短い動画を作ります。

**18歳未満は使えません。出演者は全員 21歳以上の設定です。**

このノートは Google Colab の画面の中で完結します。難しいソフトの画面は開きません。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fireworker011/Research/blob/cursor/minimax-h3-motion-identity-e959/minimax_h3_lora_studio.ipynb)

## やること（3つだけ）

1. **①** を実行 → Google Drive の許可を出す
2. **②** を実行 → 初回だけ待ちます（部品のダウンロード。2回目は速い）
3. **③** でシーンを選んで実行 → 下に動画が出る

初めてなら、シーンのフォームはいじらなくて大丈夫です。**②の「CivitaiのAPIキー」だけ貼って**、上から順に ▶ を押す。

## 準備（最初の1回）

1. 上の **Open in Colab** を開く
2. 右上の **ランタイム → ランタイムのタイプを変更 → GPU を A100**
3. **Civitai の API キー**（部品のダウンロードに使う。シークレットは不要）
   - https://civitai.com/user/account を開く → 下の **API Keys** → **Add API key** → コピー
   - **②の「CivitaiのAPIキー」欄に貼る**（このノートのフォーム。左の鍵マークは使わなくてよい）
   - キーは画面に出ません。ノートを保存する前に欄を空に戻す
4. メニュー **ランタイム → すべてのセルを実行** でも、①②③を順に押しても同じ

できた動画は Google Drive の  
`マイドライブ / minimax-h3-comfyui / output`

写真から作るときは、同じ Drive の `input` フォルダに jpg を置いてから ③ を実行。

## シーンの選び方（③で選ぶ）

| ③で選ぶ名前 | どんな動画 | 自動で入る部品 |
|---|---|---|
| 穴アップ（舐め・指） | 穴がよく見えるアップ | 穴の見え方 + アナル挿入 |
| アナル挿入 | 後ろからの挿入。穴が膣に逃げやすいとき | 総合えっち + アナル挿入 |
| ふたなりフェラ | ふたなりのフェラ。写真からの方が安定 | ふたなり + 竿 + フェラ |
| フェラ | フェラ | フェラ + 竿 |
| 騎乗位 | 上に乗って動く | 写真なら騎乗のポーズ、文章なら総合えっち |

文章欄は **空のままでOK**。おすすめの英文が自動で入ります。自分で書きたいときだけ貼る。

触らなくていい項目（リアル寄せ・胸など）は ③ のいちばん下にあります。


## ① Google Drive をつなぐ

下のセルを実行すると、許可のポップアップが出ます。**許可** を押してください。

フォームは触らなくて大丈夫です。GPU が A100 でないとここで止まります。


In [ ]:
#@title ① Drive の許可を出す（ここは触らなくてOK）
print("① Google Drive につないでいます…")

from google.colab import drive
import os

DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3-comfyui"  #@param {type:"string"}
COMFY_DIR = "/content/ComfyUI"

drive.mount("/content/drive")

DRIVE_MODELS = f"{DRIVE_ROOT}/models"
for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
    os.makedirs(f"{DRIVE_MODELS}/{sub}", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/input", exist_ok=True)

with open("/content/h3_paths.env", "w") as f:
    f.write(f"DRIVE_ROOT={DRIVE_ROOT}\n")
    f.write(f"DRIVE_MODELS={DRIVE_MODELS}\n")
    f.write(f"COMFY_DIR={COMFY_DIR}\n")

import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU がオフです。上のメニュー「ランタイム」→「ランタイムのタイプを変更」→ GPU を A100 にして、①からやり直してください。")
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
print("つながった Drive:", DRIVE_ROOT)
print("動画の保存先:", f"{DRIVE_ROOT}/output")
print("写真を置く場所:", f"{DRIVE_ROOT}/input")
print("GPU:", torch.cuda.get_device_name(0), "メモリ:", round(vram, 1), "GB")
if vram < 20:
    raise SystemExit("メモリが足りません。GPU を A100 にしてください。")
print()
print("① 完了。次は②を実行してください。初回は待ちます。")


## ② 部品を用意する（初回だけ長い）

下のセルで、動画の土台と「えっち用の部品（LoRA）」を Drive に入れます。

- **初めて** … 20〜40分かかることがあります。途中で止まっても、もう一度押せば続きから入ります
- **2回目以降** … すでに入っているファイルは飛ばすので速いです
- 初めてなら「よく使う部品を全部入れる」は **オンのまま**（③でシーンを変えても困らない）

**Civitai の API キー** は下の②セルの欄に貼ります。左の鍵（シークレット）は使わなくて大丈夫です。

1. [civitai.com のアカウント画面](https://civitai.com/user/account) を開く
2. **API Keys** → **Add API key** でキーを作ってコピー
3. ②の **CivitaiのAPIキー** 欄に貼って実行

401 / 403 が出たら、キーの貼り忘れです。欄に貼って②をもう一度。キー自体は画面に出ません。


In [ ]:
#@title ② 土台と部品を入れる（初回は待つ）
print("② 準備を始めています…")

#@markdown ### Civitai の API キー（ここに貼る。シークレット不要）
#@markdown 取り方: [civitai.com/user/account](https://civitai.com/user/account) → API Keys → Add API key
CivitaiのAPIキー = ""  #@param {type:"string"}
#@markdown **よく使う部品を全部入れる（初めてならオンのまま）**
よく使う部品を全部入れる = True  #@param {type:"boolean"}
#@markdown 全部オフにするなら、今使うシーンだけ:
今使うシーン = "穴アップ（舐め・指）"  #@param ["穴アップ（舐め・指）", "アナル挿入", "ふたなりフェラ", "フェラ", "騎乗位"]

import json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_ROOT = Path(env["DRIVE_ROOT"])
DRIVE_MODELS = Path(env["DRIVE_MODELS"])
COMFY_DIR = Path(env["COMFY_DIR"])
PORT = 8188
BRANCH = "cursor/minimax-h3-motion-identity-e959"
RAW = f"https://raw.githubusercontent.com/fireworker011/Research/{BRANCH}"
STUDIO = Path("/content/h3-lora-studio")

def sh(cmd, **kw):
    return subprocess.run(cmd, check=False, **kw)

def fetch_text(url: str, dest: Path) -> bool:
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, dest)
        return dest.is_file() and dest.stat().st_size > 100
    except Exception:
        print("ファイル取得に失敗:", dest.name)
        return False

print("説明書を取っています…")
helpers = [
    "colab/h3_r2v_core.py",
    "colab/h3_motion_graphics.py",
    "colab/h3_i2v_phone.py",
    "colab/h3_t2v.py",
    "colab/h3_lora_studio.py",
]
studio_files = [
    "h3-lora-studio/catalog/loras.json",
    "h3-lora-studio/scripts/select_loras.py",
    "h3-lora-studio/profiles/anal_closeup.json",
    "h3-lora-studio/profiles/anal_penetration.json",
    "h3-lora-studio/profiles/futa_blowjob.json",
    "h3-lora-studio/profiles/oral.json",
    "h3-lora-studio/profiles/riding.json",
]
for rel in helpers:
    dest = Path("/content") / Path(rel).name
    if not fetch_text(f"{RAW}/{rel}", dest):
        raise SystemExit("説明書の取得に失敗しました。ネットを確認して②をもう一度。")
    shutil.copy2(dest, DRIVE_ROOT / dest.name)
for rel in studio_files:
    dest = Path("/content") / rel
    if not fetch_text(f"{RAW}/{rel}", dest):
        raise SystemExit("シーン設定の取得に失敗しました。②をもう一度。")

sys.path.insert(0, "/content")
from h3_i2v_phone import i2v_download_jobs
from h3_lora_studio import (
    SITUATION_HELP, civitai_token, civitai_token_help, download_jobs_for,
    fetch_weight, load_catalog, missing_civitai_files, resolve_situation, situation_ids,
)

print("今のシーン:", 今使うシーン)
print(SITUATION_HELP[resolve_situation(今使うシーン)])
print()

if not (COMFY_DIR / "main.py").is_file():
    print("動画ソフトを入れています…")
    sh(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)])
else:
    sh(["git", "-C", str(COMFY_DIR), "pull", "--ff-only"])
req = COMFY_DIR / "requirements.txt"
if req.is_file():
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

def link_dir(link_path: Path, target: Path):
    target.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink() or link_path.is_file():
        link_path.unlink()
    elif link_path.is_dir():
        shutil.rmtree(link_path)
    link_path.symlink_to(target)

models_root = COMFY_DIR / "models"
models_root.mkdir(parents=True, exist_ok=True)
for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
    link_dir(models_root / sub, DRIVE_MODELS / sub)
link_dir(COMFY_DIR / "output", DRIVE_ROOT / "output")
link_dir(COMFY_DIR / "input", DRIVE_ROOT / "input")

print("大きな土台を入れています（すでにあれば飛ばします）…")
for url, dest in i2v_download_jobs(DRIVE_MODELS):
    if "turbo" in dest.name.lower():
        print("  高速化部品は使いません:", dest.name)
        continue
    fetch_weight(url, dest)

sid = resolve_situation(今使うシーン)
ids = situation_ids(sid)
if よく使う部品を全部入れる:
    ids = []
    for key in ("anal_closeup", "anal_penetration", "futa_blowjob", "oral", "riding"):
        ids.extend(situation_ids(key))
    print("よく使う部品を全部入れます。③でシーンを変えても大丈夫です。")
else:
    print("今のシーン用だけ入れます:", 今使うシーン)

catalog = load_catalog(STUDIO)
# Civitai API をここで読む。名前は CIVITAI_API_TOKEN。値は print しない。
token = civitai_token(CivitaiのAPIキー)
print("Civitai API:", "読み込み済み（値は出しません）" if token else "空")
jobs = download_jobs_for(ids, DRIVE_MODELS / "loras", catalog=catalog)
need = missing_civitai_files(jobs)
if need and not token:
    raise SystemExit(civitai_token_help())
for url, dest, row in jobs:
    auth = "civitai" if str(row.get("source")) == "civitai" else ""
    fetch_weight(url, dest, token=token, auth=auth)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def comfy_up() -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/object_info", timeout=3) as r:
            obj = json.loads(r.read().decode())
        return "MiniMaxH3ImageToVideo" in obj
    except Exception:
        return False

if comfy_up():
    print("動画エンジンはすでに起動しています")
else:
    print("動画エンジンを起動しています…")
    log = Path("/content/comfyui.log")
    log_f = open(log, "w", buffering=1)
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PORT),
           "--highvram", "--disable-auto-launch", "--enable-cors-header"]
    subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=log_f, stderr=subprocess.STDOUT, start_new_session=True)
    ok = False
    for _ in range(90):
        if comfy_up():
            ok = True
            break
        time.sleep(2)
    if not ok:
        print(log.read_text(errors="replace")[-2000:])
        raise SystemExit("起動に失敗しました。ランタイムを再起動して①からやり直してください。")
    print("起動できました")

print()
print("② 完了。次は③でシーンを選んで実行してください。")


## ③ 動画を作る

**シーン** と **作り方** を選んで実行します。文章は空のままで、おすすめ文が入ります。

| 作り方 | 必要なもの |
|---|---|
| テキストから（写真なし） | なし。縦動画（9:16） |
| 写真から（1枚必要） | Drive の `input` に jpg。顔や体を固定したいとき |

自分で文章を書くなら、出演者は「21歳以上の成人」と書いてください。未成年の表現は拒否されます。

おすすめ文の例（空欄のときに自動で近い内容になります）:

- **穴アップ** … 穴が見えるクローズアップ。舐め、それから指
- **アナル挿入** … 四つん這い、後ろから肛門。膣には入れない
- **ふたなりフェラ / フェラ** … 正面から竿が見える。`bl0w_j0b` と `PENISLORA` は自動で足します


In [ ]:
#@title ③ 動画を作る（ここだけ選ぶ）
#@markdown ### まずここ
やりたいシーン = "穴アップ（舐め・指）"  #@param ["穴アップ（舐め・指）", "アナル挿入", "ふたなりフェラ", "フェラ", "騎乗位"]
作り方 = "テキストから（写真なし）"  #@param ["テキストから（写真なし）", "写真から（1枚必要）"]
#@markdown 文章は空でOK（おすすめ文を自動で使います）
文章 = ""  #@param {type:"string"}
#@markdown 写真からのときだけ。`auto` なら input フォルダの一番新しい jpg
写真ファイル = "auto"  #@param {type:"string"}
秒数 = 10  #@param {type:"number"}

#@markdown ---
#@markdown ### 触らなくていい（上級）
画面の向き = "おまかせ"  #@param ["おまかせ", "縦（スマホ）", "横", "やや正方形"]
リアル寄り = False  #@param {type:"boolean"}
胸を強調 = False  #@param {type:"boolean"}
動きの底上げ = False  #@param {type:"boolean"}
静止画用の写実 = False  #@param {type:"boolean"}
試し打ちだけ = False  #@param {type:"boolean"}

print("③ 設定を読みます…")

import json, os, sys, time, uuid, urllib.request, urllib.error
from pathlib import Path
from IPython.display import display, Video, HTML

sys.path.insert(0, "/content")
sys.path.insert(0, "/content/h3-lora-studio/scripts")
from h3_r2v_core import is_oom_error, frames
from h3_i2v_phone import collect_output_videos, newest_mp4, newest_image, stage_image_into_input, is_auto_image_name
from h3_t2v import assert_t2v_graph, build_t2v_graph, canvas_for_aspect, t2v_retry_plans
from h3_motion_graphics import assert_i2va_graph, build_i2va_graph, i2va_retry_plans, CANVAS_8_9
from h3_lora_studio import (
    explain_choice, friendly_lora, inject_lora_stack, load_catalog, merge_optional,
    prepend_triggers, resolve_mode, resolve_situation,
)
from select_loras import select_loras

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
COMFY_DIR = Path(env["COMFY_DIR"])
DRIVE_ROOT = Path(env["DRIVE_ROOT"])
OUT = COMFY_DIR / "output"
PORT = 8188
STUDIO = Path("/content/h3-lora-studio")
STEPS = 16
SEED = 42
FILENAME_PREFIX = "video/h3_lora_studio"

SITUATION = resolve_situation(やりたいシーン)
MODE = resolve_mode(作り方)
PROMPT = 文章.strip() or "（シーン）"
print()
print(explain_choice(やりたいシーン, 作り方))
print()

cfg = select_loras(
    profile_name=SITUATION,
    mode=MODE,
    prompt_arg=PROMPT,
    catalog_path=STUDIO / "catalog" / "loras.json",
    profiles_dir=STUDIO / "profiles",
    turbo_override=False,
)
extras = []
if 動きの底上げ:
    extras.append("astro-nsfw-h3")
if 胸を強調:
    extras.append("tiddies-realism-slider")
if リアル寄り:
    extras.append("h3-realism-people")
if 静止画用の写実:
    extras.append("photoreal-h3-still")
    print("注意: 静止画用の写実は、穴のアップ動画向きではありません。")
stack = merge_optional(cfg["stack"], extras=extras, catalog=load_catalog(STUDIO), mode=MODE)
prompt = prepend_triggers(cfg["prompt"], stack)
if MODE == "t2v" and ("Picture 1" in prompt or "first_frame" in prompt.lower()):
    raise SystemExit("テキストから作るときは、写真ロックの文を入れません。文章欄を空にしてください。")

print("入る部品:")
for x in stack:
    print(" -", friendly_lora(x["id"]), "強さ", x.get("strength_model"))
print()
print("使う文章（先頭）:")
print(prompt[:450])
print("…")
print()

w, h = int(cfg["canvas"]["width"]), int(cfg["canvas"]["height"])
if 画面の向き == "縦（スマホ）":
    w, h = canvas_for_aspect("9:16")
elif 画面の向き == "横":
    w, h = canvas_for_aspect("16:9")
elif 画面の向き == "やや正方形":
    w, h = CANVAS_8_9
print("画面サイズ:", w, "x", h, " / 秒数:", 秒数, " / 品質ステップ:", max(int(STEPS), 16), "（速いモードは使いません）")

obj = {}
if not 試し打ちだけ:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/object_info", timeout=60) as r:
        obj = json.loads(r.read().decode())
    if "MiniMaxH3ImageToVideo" not in obj:
        raise SystemExit("エンジンがまだです。②を先に実行してください。")

diff = list((COMFY_DIR / "models/diffusion_models").glob("*fl2va*"))
if not diff and not 試し打ちだけ:
    raise SystemExit("土台がありません。②を先に実行してください。")
unet = diff[0].name if diff else "minimax_h3_fl2va_pruned_int8_convrot.safetensors"

first_name = None
if MODE == "i2v":
    inp = COMFY_DIR / "input"
    if is_auto_image_name(写真ファイル):
        hit = newest_image([DRIVE_ROOT / "input", inp])
        if hit is None:
            raise SystemExit("写真が見つかりません。スマホの Drive で「minimax-h3-comfyui/input」に jpg を置いてから、もう一度③を実行してください。")
        first_name = stage_image_into_input(hit, inp)
    else:
        src = Path(写真ファイル)
        if not src.is_file():
            src = DRIVE_ROOT / "input" / 写真ファイル
        if not src.is_file():
            raise SystemExit("その写真ファイルがありません: " + 写真ファイル)
        first_name = stage_image_into_input(src, inp)
    print("使う写真:", first_name)

plans = t2v_retry_plans(width=w, height=h) if MODE == "t2v" else i2va_retry_plans(width=w, height=h)

def make_graph(plan):
    if MODE == "t2v":
        g = build_t2v_graph(
            prompt=prompt, unet=unet, lora_name=None, lora_strength=0,
            width=int(plan["width"]), height=int(plan["height"]),
            duration_s=float(秒数), seed=int(SEED), steps=max(int(STEPS), 16),
            filename_prefix=FILENAME_PREFIX,
            has_lora_loader=("LoraLoaderModelOnly" in obj) or 試し打ちだけ,
            has_audio_decode=("VAEDecodeAudio" in obj) or 試し打ちだけ,
        )
        inject_lora_stack(g, stack, steps=max(int(STEPS), 16))
        errs = assert_t2v_graph(g)
    else:
        g = build_i2va_graph(
            first_image=first_name, last_image=None, prompt=prompt, unet=unet,
            lora_name=None, lora_strength=0,
            width=int(plan["width"]), height=int(plan["height"]),
            duration_s=float(秒数), seed=int(SEED), steps=max(int(STEPS), 16),
            filename_prefix=FILENAME_PREFIX,
            has_lora_loader=("LoraLoaderModelOnly" in obj) or 試し打ちだけ,
            has_audio_decode=("VAEDecodeAudio" in obj) or 試し打ちだけ,
        )
        inject_lora_stack(g, stack, steps=max(int(STEPS), 16))
        errs = assert_i2va_graph(g, expect_last=False)
    if errs:
        raise SystemExit(errs)
    loaders = [n for n in g.values() if n.get("class_type") == "LoraLoaderModelOnly"]
    if any("turbo" in str(n["inputs"].get("lora_name", "")).lower() for n in loaders):
        raise SystemExit("速いモードの部品が混ざったので止めています。②からやり直してください。")
    return g

def post_prompt(g):
    body = {"prompt": g, "client_id": str(uuid.uuid4())}
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/prompt",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            return json.loads(r.read().decode()), None
    except urllib.error.HTTPError as e:
        return None, e.read().decode(errors="replace")[:2000]

def wait_prompt(pid):
    for _ in range(360):
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/history/{pid}", timeout=60) as r:
            hist = json.loads(r.read().decode())
        entry = hist.get(pid)
        if entry:
            st = entry.get("status") or {}
            if st.get("completed") or entry.get("outputs"):
                if st.get("status_str") == "error":
                    return False, entry
                return True, entry
        time.sleep(2)
    return False, "timeout"

if 試し打ちだけ:
    g = make_graph(plans[0])
    print("試し打ちOK。部品:", [n["inputs"]["lora_name"] for n in g.values() if n.get("class_type") == "LoraLoaderModelOnly"])
    print("実際の動画は「試し打ちだけ」をオフにして③をもう一度。")
else:
    print()
    print("作り始めています。数分〜十数分かかることがあります…")
    last_err = None
    ok_entry = None
    before = newest_mp4(OUT)
    for plan in plans:
        print("サイズを試しています:", plan.get("label") or plan)
        g = make_graph(plan)
        res, err = post_prompt(g)
        if err:
            last_err = err
            if is_oom_error(err):
                print("メモリが足りなかったので、小さい画面でやり直します。")
                continue
            raise SystemExit("失敗しました。②からやり直すか、シーンを変えてみてください。")
        ok, payload = wait_prompt(res["prompt_id"])
        if ok:
            ok_entry = payload
            break
        last_err = payload
        if is_oom_error(str(payload)):
            print("メモリが足りなかったので、小さい画面でやり直します。")
            continue
        raise SystemExit("失敗しました。写真から作るなら input の jpg を確認してください。")
    else:
        raise SystemExit("メモリ不足で作れませんでした。秒数を 5 にするか、A100 のまま②からやり直してください。")
    videos = collect_output_videos(ok_entry, OUT)
    fresh = newest_mp4(OUT)
    if fresh and fresh not in videos and (before is None or fresh != before):
        videos.append(fresh)
    if not videos:
        print("ファイル名が取れませんでした。Drive の output フォルダを見てください:", OUT)
    else:
        print()
        print("できました。下に再生、Drive にも保存しています。")
        for p in videos:
            print("保存:", p)
            if p.is_file():
                display(HTML(f"<p style='font-size:16px'>保存先: <code>{p}</code></p>"))
                display(Video(str(p), embed=True, width=360))
print()
print("③ 完了。キーは画面に出していません。")
